# Semana 5: KNN, Naive Bayes y SVM
## Notebook Conceptual (NB1) – Experimentación con Datos Sintéticos

**Propósito:** Conocer tres algoritmos clásicos de clasificación, sus fundamentos y cuándo aplicarlos. Profundizamos en la geometría del espacio de características y el efecto de hiperparámetros.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Implementar KNN manualmente y visualizar el efecto de k.
- Entender Naive Bayes y visualizar las distribuciones condicionales.
- Visualizar el margen máximo y los vectores soporte en SVM.
- Experimentar con kernels (lineal, RBF) y sus hiperparámetros (C, gamma).
- Medir tiempos de entrenamiento y predicción para diferentes tamaños de datos.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from collections import Counter

# Scikit-learn
from sklearn.datasets import make_classification, make_moons, make_circles
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Generación de Datos Sintéticos

Creamos tres tipos de datasets en 2D para visualizar fácilmente las fronteras de decisión:
1. **Linealmente separable**
2. **Lunas (make_moons)** - no lineal
3. **Círculos concéntricos (make_circles)** - no lineal

In [ ]:
# 1. Linealmente separable
X_lin, y_lin = make_classification(n_samples=300, n_features=2, n_redundant=0, 
                                    n_clusters_per_class=1, class_sep=2.0, random_state=42)

# 2. Lunas (moons)
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=42)

# 3. Círculos concéntricos
X_circles, y_circles = make_circles(n_samples=300, noise=0.1, factor=0.3, random_state=42)

# Visualizamos los tres datasets
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(X_lin[y_lin==0, 0], X_lin[y_lin==0, 1], c='blue', label='Clase 0', alpha=0.6)
axes[0].scatter(X_lin[y_lin==1, 0], X_lin[y_lin==1, 1], c='red', label='Clase 1', alpha=0.6)
axes[0].set_title('Linealmente separable')
axes[0].legend()

axes[1].scatter(X_moons[y_moons==0, 0], X_moons[y_moons==0, 1], c='blue', alpha=0.6)
axes[1].scatter(X_moons[y_moons==1, 0], X_moons[y_moons==1, 1], c='red', alpha=0.6)
axes[1].set_title('Lunas (make_moons)')

axes[2].scatter(X_circles[y_circles==0, 0], X_circles[y_circles==0, 1], c='blue', alpha=0.6)
axes[2].scatter(X_circles[y_circles==1, 0], X_circles[y_circles==1, 1], c='red', alpha=0.6)
axes[2].set_title('Círculos concéntricos')

plt.tight_layout()
plt.show()

---
## 2. K-Vecinos más Cercanos (KNN)

### 2.1. Implementación manual

Implementamos KNN desde cero:
- Cálculo de distancia euclidiana
- Voto mayoritario de los k vecinos más cercanos

In [ ]:
def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2)**2))

class KNNManual:
    def __init__(self, k=3):
        self.k = k
    
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
    
    def predict(self, X):
        predictions = [self._predict(x) for x in X]
        return np.array(predictions)
    
    def _predict(self, x):
        # Calcular distancias a todos los puntos de entrenamiento
        distances = [euclidean_distance(x, x_train) for x_train in self.X_train]
        # Obtener índices de los k más cercanos
        k_indices = np.argsort(distances)[:self.k]
        # Obtener etiquetas de esos vecinos
        k_labels = [self.y_train[i] for i in k_indices]
        # Voto mayoritario
        most_common = Counter(k_labels).most_common(1)
        return most_common[0][0]

# Probamos en datos lineales
X_train, X_test, y_train, y_test = train_test_split(X_lin, y_lin, test_size=0.3, random_state=42)

knn_manual = KNNManual(k=5)
knn_manual.fit(X_train, y_train)
y_pred_manual = knn_manual.predict(X_test)

print(f"Precisión KNN manual (k=5): {accuracy_score(y_test, y_pred_manual):.4f}")

### 2.2. Visualización de fronteras para distintos k

Usamos KNeighborsClassifier de scikit-learn para visualizar cómo cambia la frontera con diferentes valores de k.

In [ ]:
def plot_decision_boundary(X, y, model, ax, title):
    # Crear malla
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    
    # Predecir sobre la malla
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    # Dibujar contorno y datos
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X[y==0, 0], X[y==0, 1], c='blue', edgecolors='k', alpha=0.6)
    ax.scatter(X[y==1, 0], X[y==1, 1], c='red', edgecolors='k', alpha=0.6)
    ax.set_title(title)
    ax.set_xlabel('X1')
    ax.set_ylabel('X2')

# Probamos diferentes k en datos de lunas
k_values = [1, 5, 15, 50]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for i, k in enumerate(k_values):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_moons, y_moons)
    plot_decision_boundary(X_moons, y_moons, knn, axes[i], f'k = {k}')

plt.tight_layout()
plt.show()

### 2.3. Tiempo de inferencia vs tamaño del dataset

Medimos cómo aumenta el tiempo de predicción con el número de muestras en KNN.

In [ ]:
sizes = [100, 500, 1000, 5000, 10000]
times_fit = []
times_predict = []

for n in sizes:
    # Generamos datos
    X, y = make_classification(n_samples=n, n_features=10, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # Medimos tiempo de entrenamiento (KNN no entrena realmente, pero medimos igual)
    knn = KNeighborsClassifier(n_neighbors=5)
    start = time.time()
    knn.fit(X_train, y_train)
    times_fit.append(time.time() - start)
    
    # Medimos tiempo de predicción
    start = time.time()
    knn.predict(X_test)
    times_predict.append(time.time() - start)

plt.figure(figsize=(10, 5))
plt.plot(sizes, times_fit, 'o-', label='Tiempo de entrenamiento')
plt.plot(sizes, times_predict, 's-', label='Tiempo de predicción')
plt.xlabel('Número de muestras')
plt.ylabel('Tiempo (segundos)')
plt.title('KNN: Tiempo vs Tamaño del dataset')
plt.legend()
plt.grid(True)
plt.show()

---
## 3. Naive Bayes (GaussianNB)

### 3.1. Visualización de distribuciones condicionales

GaussianNB asume que cada característica sigue una distribución normal para cada clase. Visualizamos estas distribuciones estimadas.

In [ ]:
# Entrenamos GaussianNB en datos lineales
gnb = GaussianNB()
gnb.fit(X_lin, y_lin)

# Obtenemos medias y varianzas por clase
print("Medias por clase:")
print(gnb.theta_)  # theta_ son las medias
print("\nVarianzas por clase:")
print(gnb.var_)    # var_ son las varianzas

# Visualizamos las distribuciones para la primera característica
plt.figure(figsize=(10, 5))

x_range = np.linspace(X_lin[:, 0].min(), X_lin[:, 0].max(), 200)

for clase in [0, 1]:
    media = gnb.theta_[clase, 0]
    std = np.sqrt(gnb.var_[clase, 0])
    pdf = (1 / (std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x_range - media) / std)**2)
    plt.plot(x_range, pdf, label=f'Clase {clase}', linewidth=2)

plt.xlabel('Valor de la característica X1')
plt.ylabel('Densidad de probabilidad')
plt.title('Distribuciones condicionales estimadas por GaussianNB')
plt.legend()
plt.grid(True)
plt.show()

### 3.2. Violación del supuesto de independencia

Generamos datos donde las características están correlacionadas dentro de cada clase, violando el supuesto de independencia, y observamos el efecto.

In [ ]:
# Generamos datos con correlación dentro de clases
np.random.seed(42)
n_samples = 500

# Clase 0: media [0,0], covarianza con correlación
mean0 = [0, 0]
cov0 = [[1, 0.8], [0.8, 1]]  # correlación alta
X0 = np.random.multivariate_normal(mean0, cov0, n_samples//2)
y0 = np.zeros(n_samples//2)

# Clase 1: media [2,2], covarianza con correlación
mean1 = [3, 3]
cov1 = [[1, 0.8], [0.8, 1]]
X1 = np.random.multivariate_normal(mean1, cov1, n_samples//2)
y1 = np.ones(n_samples//2)

X_corr = np.vstack([X0, X1])
y_corr = np.hstack([y0, y1])

# Visualizamos
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.scatter(X_corr[y_corr==0, 0], X_corr[y_corr==0, 1], c='blue', alpha=0.5)
plt.scatter(X_corr[y_corr==1, 0], X_corr[y_corr==1, 1], c='red', alpha=0.5)
plt.title('Datos con correlación intra-clase')
plt.xlabel('X1')
plt.ylabel('X2')

# Entrenamos GaussianNB
gnb_corr = GaussianNB()
gnb_corr.fit(X_corr, y_corr)

# Mostramos la matriz de confusión
from sklearn.metrics import confusion_matrix
y_pred_corr = gnb_corr.predict(X_corr)
cm_corr = confusion_matrix(y_corr, y_pred_corr)

plt.subplot(1, 2, 2)
sns.heatmap(cm_corr, annot=True, fmt='d', cmap='Blues')
plt.title('Matriz de confusión - GaussianNB')
plt.xlabel('Predicho')
plt.ylabel('Real')

plt.tight_layout()
plt.show()

print(f"Precisión: {accuracy_score(y_corr, y_pred_corr):.4f}")

---
## 4. Máquinas de Vectores de Soporte (SVM)

### 4.1. SVM lineal: margen máximo y vectores soporte

Visualizamos el hiperplano, el margen y los vectores soporte en un problema linealmente separable.

In [ ]:
# Usamos datos lineales
svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_lin, y_lin)

# Función para visualizar SVM lineal
def plot_svm_linear(X, y, model, ax):
    # Dibujar puntos
    ax.scatter(X[y==0, 0], X[y==0, 1], c='blue', edgecolors='k', alpha=0.6)
    ax.scatter(X[y==1, 0], X[y==1, 1], c='red', edgecolors='k', alpha=0.6)
    
    # Dibujar hiperplano
    w = model.coef_[0]
    b = model.intercept_[0]
    
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    xx = np.linspace(x_min, x_max, 100)
    yy = -(w[0] * xx + b) / w[1]
    
    # Margen
    margin = 1 / np.sqrt(np.sum(w**2))
    yy_down = yy - np.sqrt(1 + (w[0]/w[1])**2) * margin
    yy_up = yy + np.sqrt(1 + (w[0]/w[1])**2) * margin
    
    ax.plot(xx, yy, 'k-', linewidth=2, label='Hiperplano')
    ax.plot(xx, yy_down, 'k--', linewidth=1, label='Margen')
    ax.plot(xx, yy_up, 'k--', linewidth=1)
    
    # Resaltar vectores soporte
    ax.scatter(model.support_vectors_[:, 0], model.support_vectors_[:, 1],
               s=100, facecolors='none', edgecolors='green', linewidth=2, label='Vectores soporte')
    ax.set_xlim(x_min, x_max)
    ax.legend()

fig, ax = plt.subplots(1, 1, figsize=(8, 6))
plot_svm_linear(X_lin, y_lin, svm_lin, ax)
ax.set_title('SVM Lineal: margen máximo y vectores soporte')
plt.show()

print(f"Número de vectores soporte: {len(svm_lin.support_vectors_)}")

### 4.2. SVM con kernel RBF: efecto de C y gamma

Exploramos cómo los parámetros C y gamma afectan la frontera de decisión.

In [ ]:
# Usamos datos de círculos (no lineales)
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

C_values = [0.1, 1, 100]
gamma_values = [0.1, 1, 10]

for i, C in enumerate(C_values):
    for j, gamma in enumerate(gamma_values):
        svm = SVC(kernel='rbf', C=C, gamma=gamma)
        svm.fit(X_circles, y_circles)
        plot_decision_boundary(X_circles, y_circles, svm, axes[i, j], 
                               f'C={C}, gamma={gamma}')

plt.tight_layout()
plt.show()

### 4.3. Comparación SVM lineal vs RBF en datos no lineales

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# SVM lineal
svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_moons, y_moons)
plot_decision_boundary(X_moons, y_moons, svm_lin, axes[0], 'SVM Lineal')

# SVM RBF
svm_rbf = SVC(kernel='rbf', C=1.0, gamma=1.0)
svm_rbf.fit(X_moons, y_moons)
plot_decision_boundary(X_moons, y_moons, svm_rbf, axes[1], 'SVM RBF (C=1, gamma=1)')

plt.tight_layout()
plt.show()

print(f"Precisión SVM lineal: {accuracy_score(y_moons, svm_lin.predict(X_moons)):.4f}")
print(f"Precisión SVM RBF: {accuracy_score(y_moons, svm_rbf.predict(X_moons)):.4f}")

---
## 5. Complejidad Computacional: Tiempos de Entrenamiento y Predicción

Comparamos los tiempos de entrenamiento y predicción para KNN, Naive Bayes y SVM con diferentes tamaños de dataset.

In [ ]:
sizes = [100, 500, 1000, 5000]
results = []

for n in sizes:
    X, y = make_classification(n_samples=n, n_features=20, random_state=42)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
    # KNN
    knn = KNeighborsClassifier(n_neighbors=5)
    start = time.time()
    knn.fit(X_train, y_train)
    time_knn_fit = time.time() - start
    start = time.time()
    knn.predict(X_test)
    time_knn_pred = time.time() - start
    
    # Naive Bayes
    gnb = GaussianNB()
    start = time.time()
    gnb.fit(X_train, y_train)
    time_gnb_fit = time.time() - start
    start = time.time()
    gnb.predict(X_test)
    time_gnb_pred = time.time() - start
    
    # SVM (RBF)
    svm = SVC(kernel='rbf')
    start = time.time()
    svm.fit(X_train, y_train)
    time_svm_fit = time.time() - start
    start = time.time()
    svm.predict(X_test)
    time_svm_pred = time.time() - start
    
    results.append({
        'n': n,
        'KNN_fit': time_knn_fit,
        'KNN_pred': time_knn_pred,
        'GNB_fit': time_gnb_fit,
        'GNB_pred': time_gnb_pred,
        'SVM_fit': time_svm_fit,
        'SVM_pred': time_svm_pred
    })

df_times = pd.DataFrame(results)
df_times

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tiempos de entrenamiento
axes[0].plot(df_times['n'], df_times['KNN_fit'], 'o-', label='KNN')
axes[0].plot(df_times['n'], df_times['GNB_fit'], 's-', label='GaussianNB')
axes[0].plot(df_times['n'], df_times['SVM_fit'], '^-', label='SVM')
axes[0].set_xlabel('Número de muestras')
axes[0].set_ylabel('Tiempo (segundos)')
axes[0].set_title('Tiempo de Entrenamiento')
axes[0].legend()
axes[0].grid(True)

# Tiempos de predicción
axes[1].plot(df_times['n'], df_times['KNN_pred'], 'o-', label='KNN')
axes[1].plot(df_times['n'], df_times['GNB_pred'], 's-', label='GaussianNB')
axes[1].plot(df_times['n'], df_times['SVM_pred'], '^-', label='SVM')
axes[1].set_xlabel('Número de muestras')
axes[1].set_ylabel('Tiempo (segundos)')
axes[1].set_title('Tiempo de Predicción')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 6. Conclusiones

Hemos explorado tres algoritmos clásicos de clasificación:

✔️ **KNN**:
- Algoritmo basado en instancias, no paramétrico.
- k pequeño → sobreajuste, k grande → suavizado.
- Tiempo de predicción crece linealmente con el tamaño del dataset.
- Necesita escalado de características.

✔️ **Naive Bayes (GaussianNB)**:
- Modelo probabilístico que asume independencia condicional.
- Muy rápido en entrenamiento y predicción.
- Visualizamos las distribuciones normales estimadas.
- La violación del supuesto de independencia puede degradar el rendimiento, pero a menudo sigue funcionando.

✔️ **SVM**:
- Busca el hiperplano de margen máximo.
- Los vectores soporte son los puntos críticos.
- El kernel RBF permite fronteras no lineales.
- Parámetros clave: C (regularización) y gamma (influencia de puntos).
- Tiempo de entrenamiento puede ser alto para grandes datasets.

**Elección del algoritmo**: Depende del problema:
- Texto → Naive Bayes (MultinomialNB).
- Datos con muchas características y relaciones no lineales → SVM con kernel.
- Problemas pequeños y necesidad de simplicidad → KNN.

En el próximo notebook (NB2) aplicaremos estos algoritmos a datasets reales y compararemos su rendimiento.

---
**Fin del Notebook Conceptual - Semana 5**